# Near Field Potential Evaluation

After computing the density on the surface, the potential

\begin{align*}
u(x)=\int_{\Gamma}G(x,y)\,\rho(y)\,d\sigma_y, \qquad x \in \mathbb R^3 \setminus \Gamma,
\end{align*}

away from or close to the boundary needs to be evaluated.

Close to the surface, fixed order quadrature does not resolve the sharp kernel peak. FMM accelerates the resulting quadrature sum but cannot remove this quadrature error, so a local correction is required.

## Local Geometry

For a close curved triangle $T=\phi(\widehat T)$, the map $\phi$ takes the reference triangle $\widehat T$ to the physical element $T$. We project the target point onto the element and build a tangent triangle at the projected point:

$$
\widehat x_0=\operatorname*{arg\,min}_{\widehat x\in\widehat T}|x-\phi(\widehat x)|,
\qquad
x_0=\phi(\widehat x_0),
\qquad
\phi_{\mathrm{lin}}(\widehat x)
=\phi(\widehat x_0)+\phi'(\widehat x_0)(\widehat x-\widehat x_0).
$$

![Near field geometry](images/nearfield_triangle.png)


## Tangent Triangle Correction

The singular correction is computed on $\phi_{\mathrm{lin}}(\widehat T)$. Let $J$ denote the surface Jacobian of $\phi$. Pulling the element integral back to the reference triangle gives

$$
\begin{aligned}
u_T(x)
&=\int_T G(x,y)\,\rho(y)\,d\sigma_y\\
&=\int_{\widehat T}G(x,\phi(\widehat y))\,\widehat\rho(\widehat y)\,J(\widehat y)\,d\widehat y.
\end{aligned}
$$

For the Laplace and Helmholtz single layer kernels, the singular part is

$$
G_{\mathrm{sing}}(x,y)=\frac{1}{4\pi|x-y|}
$$

We add and subtract this part on the tangent triangle:

$$
\begin{aligned}
u_T(x)
&=\int_{\widehat T}G_{\mathrm{sing}}(x,\phi_{\mathrm{lin}}(\widehat y))\,
\widehat\rho(\widehat x_0)\,J(\widehat x_0)\,d\widehat y\\
&\quad+\int_{\widehat T}\Big(
G(x,\phi(\widehat y))\,\widehat\rho(\widehat y)\,J(\widehat y)
-G_{\mathrm{sing}}(x,\phi_{\mathrm{lin}}(\widehat y))\,
\widehat\rho(\widehat x_0)\,J(\widehat x_0)
\Big)d\widehat y.
\end{aligned}
$$

For the Laplace kernel this cancels the leading $1/r$ behavior and leaves a bounded remainder. For Helmholtz,

$$
G_\kappa(x,y)=\frac{e^{i\kappa|x-y|}}{4\pi|x-y|}
=\frac{1}{4\pi r}+\frac{i\kappa}{4\pi}+O(r),
\qquad r=|x-y|,
$$

so the same subtraction removes the singular part and leaves a smooth Helmholtz remainder. The tangent triangle integral is evaluated by the exact formula {cite}`GumerovDuraiswami2021`; the remainder uses standard Gauss quadrature.

## FMM Near Field Correction

Let $\mathcal N(x)$ be the set of source triangles close to the target point $x$. The FMM result already contains their ordinary quadrature contributions, which are corrected by

$$
u(x)\approx u_{\mathrm{FMM}}(x)
+\sum_{T\in\mathcal N(x)}
\left(u_T^{\mathrm{corr}}(x)-u_T^{\mathrm{quad}}(x)\right).
$$

$u_T^{\mathrm{quad}}$ is the contribution already represented by FMM, and $u_T^{\mathrm{corr}}$ includes the tangent triangle correction. The current implementation marks a triangle as near when the target distance to its center is smaller than the element size.